In [1]:
"""
Step 4: bit-accurate fixed-point model of the whole classifier + quantization sweeps + golden vectors.

Pipeline modelled (all integer arithmetic, what the FPGA would compute):
  1. DLO MAC      : acc_k = sum_t  x_int[t] * w_int[k][t]        (w_int: signed 'dlo_bits' integers)
  2. Normalizer   : tmp = acc + 2^n - mu ; u = (tmp << F) >> (n+1) ; v = u - 2^(F-1)   (v in [-0.5, 0.5])
  3. Monomials    : p = (v_i * v_j) >> F                        (truncation, like dropping LSBs in a DSP48)
  4. Dot product  : score = sum_k  w_cls_int[k] * o_int[k]       (o = [1, features, monomials])
  5. Decision     : score > thr

Float reference = identical structure with NO rounding, so any difference is purely quantization loss.

Needs step2_filters.npz from step1_2_filters_baselines.py.
"""
import os
import numpy as np

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
FILTER_FILE = "step2_filters.npz"
ALPHA_KEY = "0.05"
DESIGNS = [("lin4", 500), ("full quad", 770)]     # (classifier kind, accumulation length in bins)
BASE = dict(dlo_bits=18, F=17, w_bits=18, w_frac=12, rnd=False, clip=99.99, auto_scale=True)
N_GOLD = 2000                                      # shots exported as golden test vectors
CH = 20_000
LAMBDA_REL = 1e-6
N_SAMPLES = 770


def p(name):
    return os.path.join(DATA_DIR, name)


# ------------------------------------------------------------------ load
if not os.path.exists(FILTER_FILE):
    raise FileNotFoundError(f"{FILTER_FILE} not found. Run step1_2_filters_baselines.py to the end first.")
z = np.load(FILTER_FILE)
clf_idx = z["clf_idx"]
names = ("Wmf_I", "Wmf_Q", "Wrmf_I", "Wrmf_Q")
Wf = np.stack([z[f"{k}_a{ALPHA_KEY}"] for k in names]).astype(np.float64)   # (4, 770)

Xtr = np.load(p("0528_X_train_0_770.npy"), mmap_mode="r")
Xte_m = np.load(p("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(p("0528_y_train_0_770.npy")).astype(int)
yte = np.load(p("0528_y_test_0_770.npy")).astype(int)


def load_rows(X, idx):
    out = np.empty((len(idx), X.shape[1]), np.float32)
    for i in range(0, len(idx), CH):
        out[i:i + CH] = X[idx[i:i + CH]]
    return out


print("loading CLF and TEST sets ...")
Xclf = load_rows(Xtr, clf_idx)
yclf = ytr[clf_idx]
Xte = load_rows(Xte_m, np.arange(len(yte)))
nonint = max(np.abs(Xclf[:2000] - np.rint(Xclf[:2000])).max(), np.abs(Xte[:2000] - np.rint(Xte[:2000])).max())
print(f"max distance of samples from an integer: {nonint:.3g}  (0 means the data is already integer ADC counts)")
Xclf = np.rint(Xclf).astype(np.int16)
Xte = np.rint(Xte).astype(np.int16)
peak = int(max(np.abs(Xclf).max(), np.abs(Xte).max()))
print(f"peak |sample| = {peak}  -> needs {int(np.ceil(np.log2(peak + 1))) + 1} signed bits\n")


# ------------------------------------------------------------------ DLO MAC
def accumulate(X, w, T):
    """w: (4,770) array of real or integer-valued weights. Channels: I,Q,I,Q for MF_I, MF_Q, RMF_I, RMF_Q.
    Float64 matmul is exact for integer inputs here (all partial sums < 2^53)."""
    N = len(X)
    out = np.empty((N, 4))
    wm = w[:, :T]
    for i in range(0, N, CH):
        x = X[i:i + CH]
        I = x[:, 0:2 * T:2].astype(np.float64)
        Q = x[:, 1:2 * T:2].astype(np.float64)
        out[i:i + CH, 0] = I @ wm[0]
        out[i:i + CH, 1] = Q @ wm[1]
        out[i:i + CH, 2] = I @ wm[2]
        out[i:i + CH, 3] = Q @ wm[3]
    return out


_dlo = {}


def get_dlo(bits):
    if bits not in _dlo:
        lim = 2 ** (bits - 1) - 1
        wq = np.empty_like(Wf)
        sh = []
        for k in range(4):
            s = int(np.floor(np.log2(lim / np.abs(Wf[k]).max())))
            wq[k] = np.clip(np.rint(Wf[k] * 2.0 ** s), -lim - 1, lim)
            sh.append(s)
        _dlo[bits] = (wq, sh)
    return _dlo[bits]


_acc = {}


def get_acc(split, bits, T):
    key = (split, bits, T)
    if key not in _acc:
        _acc[key] = np.rint(accumulate(Xclf if split == "clf" else Xte, get_dlo(bits)[0], T)).astype(np.int64)
    return _acc[key]


# ------------------------------------------------------------------ feature builders
def pairs(kind):
    if kind == "lin2":
        return 2, []
    if kind == "lin4":
        return 4, []
    if kind == "spec quad":
        return 4, [(0, 0), (1, 1), (0, 1), (0, 2), (1, 3)]
    if kind == "full quad":
        return 4, [(i, j) for i in range(4) for j in range(i, 4)]
    raise ValueError(kind)


def build_feats(v, kind, mul):
    nb, pr = pairs(kind)
    cols = [v[:, i] for i in range(nb)] + [mul(v[:, i], v[:, j]) for i, j in pr]
    return np.stack(cols, 1)


def ridge(O, y):
    t = 2.0 * y - 1.0
    A = O.T @ O + LAMBDA_REL * len(O) * np.eye(O.shape[1])
    return np.linalg.solve(A, O.T @ t)


def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def fid(pred, y):
    f0 = np.mean(~pred[y == 0])
    f1 = np.mean(pred[y == 1])
    return f0, f1, 0.5 * (f0 + f1)


def paired(correct_a, correct_b):
    d = correct_a.astype(float) - correct_b.astype(float)
    return d.mean(), d.std() / np.sqrt(len(d))


# ------------------------------------------------------------------ float reference (same structure, no rounding)
def run_float(kind, T, clip=BASE["clip"]):
    ac = accumulate(Xclf, Wf, T)
    at = accumulate(Xte, Wf, T)
    mu = ac.mean(0)
    A = np.percentile(np.abs(ac - mu), clip, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1e-12)))

    def feat(a):
        return np.clip((a + 2.0 ** n - mu) / 2.0 ** (n + 1), 0, 1) - 0.5

    mul = lambda a, b: a * b
    Oc = np.c_[np.ones(len(ac)), build_feats(feat(ac), kind, mul)]
    Ot = np.c_[np.ones(len(at)), build_feats(feat(at), kind, mul)]
    w = ridge(Oc, yclf)
    thr = best_threshold(Oc @ w, yclf)
    pred = (Ot @ w) > thr
    f0, f1, F = fid(pred, yte)
    return F, f0, f1, pred == (yte == 1)


# ------------------------------------------------------------------ fixed-point model
def norm_consts(acc_c, clip):
    mu = np.rint(acc_c.mean(0)).astype(np.int64)
    A = np.percentile(np.abs(acc_c - mu), clip, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1.0))).astype(np.int64)
    return mu, n


def normalize(acc, mu, n, F):
    assert int(n.max()) + 1 + F < 62, "normalizer would overflow int64 in this model"
    one = np.int64(1)
    tmp = acc + (one << n) - mu
    tmp = np.clip(tmp, 0, one << (n + 1))          # saturate (hardware: two comparators)
    u = (tmp << F) >> (n + 1)                       # in [0, 2^F]   (= [0, 1.0])
    return u - (one << (F - 1))                     # centered: [-0.5, +0.5]


def int_feats(v, kind, cfg):
    F = cfg["F"]
    if cfg["rnd"]:
        mul = lambda a, b: (a * b + (1 << (F - 1))) >> F
    else:
        mul = lambda a, b: (a * b) >> F
    O = build_feats(v, kind, mul)
    return np.c_[np.full(len(v), 1 << F, dtype=np.int64), O]


def fit_fixed(kind, T, cfg):
    acc_c = get_acc("clf", cfg["dlo_bits"], T)
    mu, n = norm_consts(acc_c, cfg["clip"])
    Oc = int_feats(normalize(acc_c, mu, n, cfg["F"]), kind, cfg)
    wf = ridge(Oc / 2.0 ** cfg["F"], yclf)
    lim = 2 ** (cfg["w_bits"] - 1)
    scale = 1.0
    if cfg["auto_scale"]:     # positive scaling of weights and threshold never changes the decision
        rng_max = (lim - 1) / 2.0 ** cfg["w_frac"]
        scale = 2.0 ** np.floor(np.log2(rng_max / np.abs(wf).max()))
        wf = wf * scale
    wraw = wf * 2.0 ** cfg["w_frac"]
    sat = int((np.abs(wraw) > lim - 1).sum())
    wq = np.clip(np.rint(wraw), -lim, lim - 1).astype(np.int64)
    sc = Oc @ wq
    thr = int(round(best_threshold(sc.astype(np.float64), yclf)))
    return dict(mu=mu, n=n, wq=wq, thr=thr, sat=sat, maxw=float(np.abs(wf).max()), scale=float(scale))


def apply_fixed(model, kind, cfg, acc):
    v = normalize(acc, model["mu"], model["n"], cfg["F"])
    O = int_feats(v, kind, cfg)
    sc = O @ model["wq"]
    return sc, sc > model["thr"], O, v


def eval_cfg(kind, T, cfg):
    model = fit_fixed(kind, T, cfg)
    sc, pred, O, v = apply_fixed(model, kind, cfg, get_acc("te", cfg["dlo_bits"], T))
    f0, f1, F = fid(pred, yte)
    return F, f0, f1, pred == (yte == 1), model


# ------------------------------------------------------------------ 1. float vs fixed at spec formats
print("=== Float reference vs fixed-point at the spec formats ===")
print(f"base config: {BASE}")
refs, base_res = {}, {}
for kind, T in DESIGNS:
    refs[(kind, T)] = run_float(kind, T)
    base_res[(kind, T)] = eval_cfg(kind, T, BASE)
    Ff, _, _, cf = refs[(kind, T)]
    Fq, f0, f1, cq, m = base_res[(kind, T)]
    d, se = paired(cq, cf)
    acc_c, acc_t = get_acc("clf", 18, T), get_acc("te", 18, T)
    accbits = int(np.ceil(np.log2(max(np.abs(acc_c).max(), np.abs(acc_t).max()) + 1))) + 1
    print(f"\n[{kind}, T={T}]  float F={Ff:.4f}   fixed F={Fq:.4f} (F0={f0:.4f}, F1={f1:.4f})   "
          f"fixed-minus-float = {100 * d:+.3f} +/- {100 * se:.3f} points")
    print(f"   DLO accumulator needs {accbits} signed bits (spec says 32; DSP48E2 accumulator is 48)")
    print(f"   normalizer n per feature = {m['n'].tolist()},  mu = {m['mu'].tolist()}")
    print(f"   classifier: {len(m['wq'])} weights, {m['sat']} saturated, max|w| after scaling = {m['maxw']:.2f}")


# ------------------------------------------------------------------ 2. one-at-a-time sweeps
sweeps = [("dlo_bits", [6, 8, 10, 12, 14, 16, 18], None),
          ("F (feature frac bits)", [6, 8, 10, 12, 14, 17], "F"),
          ("w_bits (6 int bits kept)", [8, 10, 12, 14, 16, 18], "w_bits"),
          ("clip percentile", [100, 99.99, 99.9], "clip")]


def run_row(label, **chg):
    cfg = dict(BASE)
    cfg.update(chg)
    if "w_bits" in chg:
        cfg["w_frac"] = chg["w_bits"] - 6
    cells = []
    for kind, T in DESIGNS:
        Fq = eval_cfg(kind, T, cfg)[0]
        Ff = refs[(kind, T)][0]
        cells.append(f"{Fq:.4f} ({100 * (Fq - Ff):+.2f})")
    print(f"{label:<34s} " + "   ".join(f"{c:>16s}" for c in cells))


print("\n=== Quantization sweeps (fidelity, and change vs float in points). Other settings = BASE ===")
print(f"{'setting':<34s} " + "   ".join(f"{k + ' T=' + str(T):>16s}" for k, T in DESIGNS))
for key in (("dlo_bits", [6, 8, 10, 12, 14, 16, 18]),):
    for v in key[1]:
        run_row(f"DLO weight bits = {v}", dlo_bits=v)
for v in [6, 8, 10, 12, 14, 17]:
    run_row(f"feature frac bits F = {v}", F=v)
for v in [8, 10, 12, 14, 16, 18]:
    run_row(f"classifier weight bits = {v}", w_bits=v)
for v in [100, 99.99, 99.9]:
    run_row(f"normalizer clip percentile = {v}", clip=v)
run_row("product rounding instead of floor", rnd=True)
run_row("NO weight auto-scaling (spec literal)", auto_scale=False)
print("\nRule of thumb: a setting is 'safe' if its change vs float is within about -0.05 points.")

# ------------------------------------------------------------------ 3. golden vectors for the RTL testbench
gold = {}
for tag, (kind, T) in zip("AB", DESIGNS):
    F, _, _, _, model = base_res[(kind, T)]
    acc = get_acc("te", BASE["dlo_bits"], T)[:N_GOLD]
    sc, pred, O, v = apply_fixed(model, kind, BASE, acc)
    wq, sh = get_dlo(BASE["dlo_bits"])
    gold.update({
        f"{tag}_x": Xte[:N_GOLD], f"{tag}_label": yte[:N_GOLD],
        f"{tag}_acc": acc, f"{tag}_v": v, f"{tag}_o": O, f"{tag}_score": sc, f"{tag}_pred": pred.astype(np.int8),
        f"{tag}_dlo_w": wq[:, :T].astype(np.int32), f"{tag}_dlo_shift": np.array(sh),
        f"{tag}_mu": model["mu"], f"{tag}_n": model["n"], f"{tag}_cls_w": model["wq"],
        f"{tag}_thr": np.int64(model["thr"]), f"{tag}_T": np.int64(T), f"{tag}_F": np.int64(BASE["F"]),
        f"{tag}_kind": np.array(kind),
    })
np.savez("golden_vectors.npz", **gold)
print(f"\nsaved golden_vectors.npz  (designs A={DESIGNS[0]}, B={DESIGNS[1]}, {N_GOLD} test shots each)")
print("An RTL testbench must reproduce *_acc, *_v, *_o, *_score and *_pred bit-for-bit.")

loading CLF and TEST sets ...
max distance of samples from an integer: 0  (0 means the data is already integer ADC counts)
peak |sample| = 4685  -> needs 14 signed bits

=== Float reference vs fixed-point at the spec formats ===
base config: {'dlo_bits': 18, 'F': 17, 'w_bits': 18, 'w_frac': 12, 'rnd': False, 'clip': 99.99, 'auto_scale': True}

[lin4, T=500]  float F=0.9610   fixed F=0.9610 (F0=0.9767, F1=0.9454)   fixed-minus-float = +0.000 +/- 0.000 points
   DLO accumulator needs 36 signed bits (spec says 32; DSP48E2 accumulator is 48)
   normalizer n per feature = [34, 34, 34, 34],  mu = [2405105367, -924020300, 1490920772, -982523547]
   classifier: 5 weights, 0 saturated, max|w| after scaling = 17.21

[full quad, T=770]  float F=0.9615   fixed F=0.9615 (F0=0.9756, F1=0.9475)   fixed-minus-float = -0.001 +/- 0.001 points
   DLO accumulator needs 37 signed bits (spec says 32; DSP48E2 accumulator is 48)
   normalizer n per feature = [35, 34, 33, 34],  mu = [4024856529, -1520690249, 1